In [1]:
import os, sys, shutil
import torch
from tqdm import tqdm
import numpy as np
import pandas as pd
import matplotlib as plt
from PIL import Image
sys.path.append("../src")
from utils import *
# from autoencoder import AutoEncoder
from gnet8_encoder import GNet8_Encoder
from matplotlib.lines import Line2D
import matplotlib as mpl
import math
import matplotlib.image as mpimg
import random
os.chdir("..")

/export/raid1/home/kneel027/mamba/envs/SS/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: '/export/raid1/home/kneel027/mamba/envs/SS/lib/python3.10/site-packages/torchvision/image.so: undefined symbol: _ZN3c1017RegisterOperatorsD1Ev'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
/export/raid1/home/kneel027/mamba/envs/SS/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
def get_kastner_rois(subject, data_path):
    kastner_labels = f"{data_path}/nsddata/freesurfer/subj0{subject}/label/Kastner2015.mgz.ctab"
    brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
    brainmask_inflated = np.nan_to_num(brainmask_inflated)
    brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
    kastner_mask = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/Kastner2015.nii.gz").get_fdata()
    kastner_mask = kastner_mask[brainmask_inflated]
    with open(kastner_labels, 'r') as file:
        labels = file.read().splitlines()
    kastner_mask_labeled = {}
    for label in labels[1:]:
        label = label.split(" ")
        kastner_mask_labeled[label[1].strip()] = np.where(kastner_mask==int(label[0]), True, False)
    kastner_mask_labeled["V1"] = np.logical_or(kastner_mask_labeled["V1v"], kastner_mask_labeled["V1d"])
    kastner_mask_labeled["V2"] = np.logical_or(kastner_mask_labeled["V2v"], kastner_mask_labeled["V2d"])
    kastner_mask_labeled["V3"] = np.logical_or(kastner_mask_labeled["V3v"], kastner_mask_labeled["V3d"])
    del kastner_mask_labeled["V1v"]
    del kastner_mask_labeled["V1d"]
    del kastner_mask_labeled["V2v"]
    del kastner_mask_labeled["V2d"]
    del kastner_mask_labeled["V3v"]
    del kastner_mask_labeled["V3d"]
    nsd_general = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/nsdgeneral.nii.gz").get_fdata()
    nsd_general = np.nan_to_num(nsd_general)
    nsd_general = nsd_general[brainmask_inflated]
    nsd_general = np.where(nsd_general==1.0, True, False)
    kastner_mask_labeled['nsd_general'] = nsd_general
    return kastner_mask_labeled

def get_hcp_mmp1_rois(subject, data_path):
    mmp1_labels = f"{data_path}/nsddata/freesurfer/subj0{subject}/label/HCP_MMP1.mgz.ctab"
    brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
    brainmask_inflated = np.nan_to_num(brainmask_inflated)
    brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
    mmp1_mask = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/HCP_MMP1.nii.gz").get_fdata()
    mmp1_mask = mmp1_mask[brainmask_inflated]
    with open(mmp1_labels, 'r') as file:
        labels = file.read().splitlines()
    mmp1_mask_labeled = {}
    for label in labels[1:]:
        label = label.split(" ")
        mmp1_mask_labeled[label[1].strip()] = np.where(mmp1_mask==int(label[0]), True, False)
    nsd_general = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/nsdgeneral.nii.gz").get_fdata()
    nsd_general = np.nan_to_num(nsd_general)
    nsd_general = nsd_general[brainmask_inflated]
    nsd_general = np.where(nsd_general==1.0, True, False)
    mmp1_mask_labeled['nsd_general'] = nsd_general
    return mmp1_mask_labeled

def get_thalamus_rois(subject, data_path):
    thalamus_labels = f"{data_path}/nsddata/templates/thalamus.ctab"
    brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
    brainmask_inflated = np.nan_to_num(brainmask_inflated)
    brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
    thalamus_mask = nib.load(f"data/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/thalamus.nii.gz").get_fdata()
    thalamus_mask = thalamus_mask[brainmask_inflated]
    with open(thalamus_labels, 'r') as file:
        labels = file.read().splitlines()
    thalamus_mask_labeled = {}
    for label in labels[1:]:
        label = label.split(" ")
        thalamus_mask_labeled[label[1].strip()] = np.where(thalamus_mask==int(label[0]), True, False)
    return thalamus_mask_labeled

def get_mtl_rois(subject, data_path):
    mtl_labels = f"{data_path}/nsddata/templates/MTL.ctab"
    brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
    brainmask_inflated = np.nan_to_num(brainmask_inflated)
    brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
    mtl_mask = nib.load(f"data/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/MTL.nii.gz").get_fdata()
    mtl_mask = mtl_mask[brainmask_inflated]
    with open(mtl_labels, 'r') as file:
        labels = file.read().splitlines()
    mtl_mask_labeled = {}
    for label in labels[1:]:
        label = label.split(" ")
        mtl_mask_labeled[label[1].strip()] = np.where(mtl_mask==int(label[0]), True, False)
    return mtl_mask_labeled

def get_streams_rois(subject, data_path):
    streams_labels = f"{data_path}/nsddata/freesurfer/subj0{subject}/label/streams.mgz.ctab"
    brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
    brainmask_inflated = np.nan_to_num(brainmask_inflated)
    brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
    streams_mask = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/streams.nii.gz").get_fdata()
    streams_mask = streams_mask[brainmask_inflated]
    with open(streams_labels, 'r') as file:
        labels = file.read().splitlines()
    streams_mask_labeled = {}
    for label in labels[1:]:
        label = label.split(" ")
        streams_mask_labeled[label[1].strip()] = np.where(streams_mask==int(label[0]), True, False)
    nsd_general = nib.load(f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/nsdgeneral.nii.gz").get_fdata()
    nsd_general = np.nan_to_num(nsd_general)
    nsd_general = nsd_general[brainmask_inflated]
    nsd_general = np.where(nsd_general==1.0, True, False)
    streams_mask_labeled['nsd_general'] = nsd_general
    return streams_mask_labeled

# kastner_mask_labeled = get_kastner_rois(1, "data/")
# print(kastner_mask_labeled.keys(), np.unique(kastner_mask_labeled['V1'], return_counts=True))

# mm1_mask_labeled = get_hcp_mmp1_rois(1, "data/")
# print(mm1_mask_labeled.keys(), np.unique(mm1_mask_labeled['V1'], return_counts=True))

# thalamus_mask_labeled = get_thalamus_rois(1, "data/")
# print(thalamus_mask_labeled.keys(), np.unique(thalamus_mask_labeled['LGN'], return_counts=True))

# mtl_mask_labeled = get_mtl_rois(1, "data/")
# print(mtl_mask_labeled.keys(), np.unique(mtl_mask_labeled['ERC'], return_counts=True))

streams_mask_labeled = get_streams_rois(1, "data/")
print(streams_mask_labeled.keys(), np.unique(streams_mask_labeled['ventral'], return_counts=True))


dict_keys(['early', 'midventral', 'midlateral', 'midparietal', 'ventral', 'lateral', 'parietal', 'nsd_general']) (array([False,  True]), array([230904,   7604]))


In [10]:
with h5py.File("/home/naxos2-raid25/kneel027/home/kneel027/data/mindeye_imagery/kastner_rois.hdf5", 'w') as hdf:
    for subject in range(1, 9):
        subject_str = f"subj{subject:02d}"
        mask_dictionary = get_kastner_masks(subject, "data/")

        # Create a group for each subject
        subject_group = hdf.create_group(subject_str)

        # Iterate through the mask dictionary and create datasets for each mask
        for region, mask in mask_dictionary.items():
            subject_group.create_dataset(region, data=mask)
            


In [ ]:
with h5py.File("/home/naxos2-raid25/kneel027/home/kneel027/data/mindeye_imagery/hcp_mmp1_rois.hdf5", 'w') as hdf:
    for subject in range(1, 9):
        subject_str = f"subj{subject:02d}"
        mask_dictionary = get_hcp_mmp1_rois(subject, "data/")

        # Create a group for each subject
        subject_group = hdf.create_group(subject_str)

        # Iterate through the mask dictionary and create datasets for each mask
        for region, mask in mask_dictionary.items():
            subject_group.create_dataset(region, data=mask)

In [16]:
with h5py.File("/home/naxos2-raid25/kneel027/home/kneel027/data/mindeye_imagery/thalamus_rois.hdf5", 'w') as hdf:
    for subject in range(1, 9):
        subject_str = f"subj{subject:02d}"
        mask_dictionary = get_thalamus_rois(subject, "data/")

        # Create a group for each subject
        subject_group = hdf.create_group(subject_str)

        # Iterate through the mask dictionary and create datasets for each mask
        for region, mask in mask_dictionary.items():
            subject_group.create_dataset(region, data=mask)

In [ ]:
with h5py.File("/home/naxos2-raid25/kneel027/home/kneel027/data/mindeye_imagery/mtl_rois.hdf5", 'w') as hdf:
    for subject in range(1, 9):
        subject_str = f"subj{subject:02d}"
        mask_dictionary = get_mtl_rois(subject, "data/")

        # Create a group for each subject
        subject_group = hdf.create_group(subject_str)

        # Iterate through the mask dictionary and create datasets for each mask
        for region, mask in mask_dictionary.items():
            subject_group.create_dataset(region, data=mask)

In [ ]:
with h5py.File("/home/naxos2-raid25/kneel027/home/kneel027/data/mindeye_imagery/streams_rois.hdf5", 'w') as hdf:
    for subject in range(1, 9):
        subject_str = f"subj{subject:02d}"
        mask_dictionary = get_streams_rois(subject, "data/")

        # Create a group for each subject
        subject_group = hdf.create_group(subject_str)

        # Iterate through the mask dictionary and create datasets for each mask
        for region, mask in mask_dictionary.items():
            subject_group.create_dataset(region, data=mask)

In [2]:
# Making large collection of labeled masks in raw brain format:
data_path = "data/"
all_masks = {}
for subject in range(1, 9):
    all_masks[f"subj{subject:02d}"] = {}
    for mask_file, label_file in zip([f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/nsdgeneral.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/Kastner2015.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/thalamus.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/MTL.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/streams.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-bodies.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-faces.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-places.nii.gz",
                                      f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-words.nii.gz"],
                                     [f"{data_path}/nsddata/freesurfer/subj0{subject}/label/nsdgeneral.mgz.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/Kastner2015.mgz.ctab",
                                      f"{data_path}/nsddata/templates/thalamus.ctab",
                                      f"{data_path}/nsddata/templates/MTL.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/streams.mgz.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-bodies.mgz.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-faces.mgz.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-places.mgz.ctab",
                                      f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-words.mgz.ctab"]):
        
        mask = nib.load(mask_file).get_fdata()
        with open(label_file, 'r') as file:
            labels = file.read().splitlines()
        for label in labels[1:]:
            label = label.split(" ")
            all_masks[f"subj{subject:02d}"][label[1].strip()] = np.where(mask==int(label[0]), True, False)
    all_masks[f"subj{subject:02d}"]["V1"] = np.logical_or(all_masks[f"subj{subject:02d}"]["V1v"], all_masks[f"subj{subject:02d}"]["V1d"])
    all_masks[f"subj{subject:02d}"]["V2"] = np.logical_or(all_masks[f"subj{subject:02d}"]["V2v"], all_masks[f"subj{subject:02d}"]["V2d"])
    all_masks[f"subj{subject:02d}"]["V3"] = np.logical_or(all_masks[f"subj{subject:02d}"]["V3v"], all_masks[f"subj{subject:02d}"]["V3d"])
    del all_masks[f"subj{subject:02d}"]["V1v"]
    del all_masks[f"subj{subject:02d}"]["V1d"]
    del all_masks[f"subj{subject:02d}"]["V2v"]
    del all_masks[f"subj{subject:02d}"]["V2d"]
    del all_masks[f"subj{subject:02d}"]["V3v"]
    del all_masks[f"subj{subject:02d}"]["V3d"]


In [22]:
import numpy as np
import nibabel as nib
from collections import defaultdict
import os

# Define the data path
data_path = "data/"

# Initialize the all_masks dictionary
all_masks = {}

# Loop through each subject
for subject in range(1, 9):
    subj_key = f"subj{subject:02d}"
    all_masks[subj_key] = {}
    print(f"\nLoading masks for {subj_key}...")

    # Define mask and label file paths
    mask_files = [
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/nsdgeneral.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/Kastner2015.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/thalamus.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/MTL.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/streams.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-bodies.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-faces.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-places.nii.gz",
        f"{data_path}/nsddata/ppdata/subj0{subject}/func1pt8mm/roi/floc-words.nii.gz"
    ]

    label_files = [
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/nsdgeneral.mgz.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/Kastner2015.mgz.ctab",
        f"{data_path}/nsddata/templates/thalamus.ctab",
        f"{data_path}/nsddata/templates/MTL.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/streams.mgz.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-bodies.mgz.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-faces.mgz.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-places.mgz.ctab",
        f"{data_path}/nsddata/freesurfer/subj0{subject}/label/floc-words.mgz.ctab"
    ]

    # Iterate through each mask and corresponding label file
    for mask_file, label_file in zip(mask_files, label_files):
        # Load the mask image
        try:
            mask_img = nib.load(mask_file)
            mask_data = mask_img.get_fdata()
            # print(f"  Loaded mask from {mask_file} with shape {mask_data.shape} and dtype {mask_data.dtype}")
        except Exception as e:
            print(f"  Error loading mask file {mask_file}: {e}")
            continue

        # Load and parse the label file
        try:
            with open(label_file, 'r') as file:
                labels = file.read().splitlines()
            # print(f"  Loaded labels from {label_file} with {len(labels)} entries.")
        except Exception as e:
            print(f"  Error loading label file {label_file}: {e}")
            continue

        # Determine if the first line is a header
        if labels and not labels[0].strip()[0].isdigit():
            label_lines = labels[1:]
            print(f"    Detected header in {label_file}; skipping the first line.")
        else:
            label_lines = labels

        # Iterate through each label entry
        for label in label_lines:
            parts = label.split()
            if len(parts) < 2:
                print(f"    Skipping malformed label line: '{label}'")
                continue
            try:
                label_id = int(parts[0])
            except ValueError:
                print(f"    Invalid label ID in line: '{label}'")
                continue
            label_name = parts[1].strip()

            # **Modification: Skip the 'Unknown' ROI**
            if label_name.lower() == 'unknown' or label_name.lower() == "unlabeled":
                # print(f"    Skipping 'Unknown' ROI (ID: {label_id}).")
                continue  # Do not add 'Unknown' to all_masks

            # Create a boolean mask for the current ROI
            roi_mask = mask_data == label_id
            voxel_count = np.sum(roi_mask)
            # print(f"    ROI '{label_name}' (ID: {label_id}) has {voxel_count} voxels.")

            # Add the ROI mask to the all_masks dictionary
            all_masks[subj_key][label_name] = roi_mask

    # **Combine V1, V2, V3 Variants**
    for area in ['V1', 'V2', 'V3']:
        combined_key = area
        variant_keys = [f"{area}v", f"{area}d"]
        combined_mask = np.logical_or.reduce([
            all_masks[subj_key].get(k, False) for k in variant_keys
        ])
        all_masks[subj_key][combined_key] = combined_mask
        # Remove variant masks
        for k in variant_keys:
            if k in all_masks[subj_key]:
                del all_masks[subj_key][k]
        voxel_count = np.sum(combined_mask)
        # print(f"  Combined ROI '{combined_key}' has {voxel_count} voxels.")

# **Processing all_masks to Create Combined Brain Masks**
combined_masks = {}  # To store the combined masks and label lists for each subject

for subj_key, rois in all_masks.items():
    print(f"\nProcessing {subj_key}...")

    # Step 1: Sort ROIs by the number of True voxels (smallest to largest)
    roi_counts = {roi: np.sum(mask) for roi, mask in rois.items()}

    # Print ROI counts for debugging
    print("  ROI voxel counts (sorted):")
    sorted_roi_counts = sorted(roi_counts.items(), key=lambda x: x[1])
    for roi, count in sorted_roi_counts:
        print(f"    {roi}: {count}")

    # Sort ROIs based on voxel counts
    sorted_rois = sorted(rois.items(), key=lambda item: roi_counts[item[0]])

    # Move 'nsdgeneral' to the top if it exists
    sorted_rois = [item for item in sorted_rois if item[0] != 'nsdgeneral']
    if 'nsdgeneral' in rois:
        sorted_rois.insert(0, ('nsdgeneral', rois['nsdgeneral']))

    # Initialize the combined mask with zeros (integer type)
    mask_shape = next(iter(rois.values())).shape
    combined_mask = np.zeros(mask_shape, dtype=np.int32)

    # Initialize the label list with '0' for no mask
    label_list = ['No Mask']

    current_label = 1  # Start labeling from 1

    for roi_name, roi_mask in sorted_rois:
        # Find voxels that are True in the current ROI and not yet assigned
        new_voxels = np.logical_and(roi_mask, combined_mask == 0)
        num_new_voxels = np.sum(new_voxels)

        print(f"  Assigning ROI '{roi_name}' with {num_new_voxels} new voxels.")

        if num_new_voxels > 10:
            # Assign the current label to these voxels
            combined_mask[new_voxels] = current_label
            # Add the ROI name to the label list
            label_list.append(roi_name)
            current_label += 1
        else:
            print(f"    ROI '{roi_name}' has no new voxels to assign.")

    # Store the combined mask and label list for the subject
    combined_masks[subj_key] = {
        'mask': combined_mask,
        'labels': label_list
    }

    print(f"  Combined mask for {subj_key} has {current_label - 1} ROIs assigned.")

# **Saving the Combined Masks and Labels to Disk**
output_path = "/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/"
os.makedirs(output_path, exist_ok=True)

for subj_key, data in combined_masks.items():
    print(f"\nSaving combined mask and labels for {subj_key}...")
    # Save the combined mask as a NIfTI file
    combined_nifti = nib.Nifti1Image(data['mask'], affine=np.eye(4))
    output_file = os.path.join(output_path, f"{subj_key}_combined_mask_nsd_general.nii.gz")
    nib.save(combined_nifti, output_file)
    print(f"  Saved combined mask to {output_file}")

    # Save the label list to a text file
    label_file = os.path.join(output_path, f"{subj_key}_labels_nsd_general.txt")
    with open(label_file, 'w') as f:
        for idx, label in enumerate(data['labels']):
            f.write(f"{idx}: {label}\n")
    print(f"  Saved labels to {label_file}")

print("\nAll combined masks and label lists have been saved.")



Loading masks for subj01...

Loading masks for subj02...

Loading masks for subj03...

Loading masks for subj04...

Loading masks for subj05...

Loading masks for subj06...

Loading masks for subj07...

Loading masks for subj08...

Processing subj01...
  ROI voxel counts (sorted):
    mTL-bodies: 0
    mTL-faces: 0
    IPS5: 14
    CA2: 17
    SC: 21
    dorsomedialPul: 22
    dorsolateralPul: 24
    CA3: 32
    TO2: 55
    LGN: 55
    IPS4: 65
    FEF: 72
    ventralPul: 94
    mTL-words: 122
    aTL-faces: 159
    SPL1: 164
    mfs-words: 165
    PHC2: 183
    LO2: 193
    PHC1: 199
    VO1: 216
    PHC: 221
    35: 241
    CA1: 242
    HT: 249
    DG: 254
    TO1: 284
    FFA-2: 310
    VO2: 313
    V3B: 315
    LO1: 343
    ERC: 345
    SUB: 350
    OFA: 355
    FBA-1: 396
    FBA-2: 430
    IPS1: 462
    OWFA: 464
    hV4: 473
    FFA-1: 484
    IPS2: 500
    VWFA-2: 505
    IPS3: 508
    RSC: 566
    IPS0: 606
    36: 652
    V3A: 696
    VWFA-1: 773
    midlateral: 834
    midp

In [21]:
mask = nib.load("/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/subj04_combined_mask_nsd_general.nii.gz").get_fdata()
brainmask_inflated = nib.load(f"{data_path}/nsddata/ppdata/subj04/func1pt8mm/roi/brainmask_inflated_1.0.nii").get_fdata()
brainmask_inflated = np.nan_to_num(brainmask_inflated)
brainmask_inflated = np.where(brainmask_inflated==1.0, True, False)
mask = mask[brainmask_inflated]
IPS5_mask = mask == 39
print(np.unique(IPS5_mask, return_counts=True))

(array([False,  True]), array([229471,    171]))


In [12]:
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
from matplotlib.colors import ListedColormap, to_rgba

def visualize_combined_mask(combined_mask_path, label_list_path, num_slices=25, grid_size=(5,5), output_file=None, dpi=600):
    """
    Visualizes a combined brain mask by extracting horizontal slices and displaying them in a grid.
    
    Parameters:
    - combined_mask_path (str): Path to the combined mask NIfTI file.
    - label_list_path (str): Path to the label list text file.
    - num_slices (int): Number of horizontal slices to extract. Default is 25.
    - grid_size (tuple): Grid size for plotting (rows, columns). Default is (5,5).
    - output_file (str): If provided, saves the figure to the specified path. Use formats like PNG, TIFF, or PDF for high quality.
    - dpi (int): Dots per inch for the saved figure. Higher values yield higher resolution. Default is 600.
    
    Returns:
    - None
    """
    
    # ----------------------------
    # Step 1: Load Combined Mask
    # ----------------------------
    try:
        mask_img = nib.load(combined_mask_path)
        mask_data = mask_img.get_fdata().astype(int)  # Ensure integer type for labels
        print(f"Loaded combined mask from '{combined_mask_path}' with shape {mask_data.shape}.")
    except Exception as e:
        print(f"Error loading combined mask file '{combined_mask_path}': {e}")
        return
    
    # ----------------------------
    # Step 2: Load Label List
    # ----------------------------
    try:
        with open(label_list_path, 'r') as f:
            labels = f.read().splitlines()
        # Parse labels into a list where index corresponds to label value
        label_list = []
        for line in labels:
            parts = line.split(':')
            if len(parts) != 2:
                print(f"Malformed label line: '{line}'. Skipping.")
                label_list.append('Unknown')
                continue
            label_name = parts[1].strip()
            label_list.append(label_name)
        print(f"Loaded {len(label_list)} labels from '{label_list_path}'.")
    except Exception as e:
        print(f"Error loading label list file '{label_list_path}': {e}")
        return
    
    # ----------------------------
    # Step 3: Assign Colors to ROIs
    # ----------------------------
    # Exclude 'No Mask' (label 0) from color assignment
    unique_labels = np.unique(mask_data)
    unique_labels = unique_labels[unique_labels != 0]  # Exclude label 0
    
    # Generate colors using a colormap
    # Using 'tab20', 'tab20b', 'tab20c' for distinct colors
    base_cmap = plt.get_cmap('tab20')
    extended_cmap = plt.get_cmap('tab20b')
    all_cmap = np.vstack((base_cmap.colors, extended_cmap.colors))
    
    num_colors_available = len(all_cmap)
    num_rois = len(unique_labels)
    
    if num_rois > num_colors_available:
        # If more ROIs than available colors, repeat the colormap
        repeats = int(np.ceil(num_rois / num_colors_available))
        color_list = (all_cmap.tolist() * repeats)[:num_rois]
        print(f"Assigned {num_rois} ROIs with repeated colors from the colormap.")
    else:
        color_list = all_cmap[:num_rois].tolist()
        print(f"Assigned {num_rois} ROIs with unique colors from the colormap.")
    
    # Create a dictionary mapping label values to colors
    label_to_color = {label: color for label, color in zip(unique_labels, color_list)}
    
    # ----------------------------
    # Step 4: Extract Slices
    # ----------------------------
    # Determine slice indices evenly spaced along the axial (z) axis
    z_dim = mask_data.shape[2]
    slice_indices = np.linspace(10, z_dim - 10, num_slices, dtype=int)
    print(f"Extracting {num_slices} slices at indices: {slice_indices}")
    
    # ----------------------------
    # Step 5: Plotting
    # ----------------------------
    # Increase the figure size for higher resolution
    # Each subplot will be larger to capture more detail
    subplot_width = 10  # inches
    subplot_height = 10  # inches
    total_width = grid_size[1] * subplot_width
    total_height = grid_size[0] * subplot_height
    
    fig, axes = plt.subplots(grid_size[0], grid_size[1], figsize=(total_width, total_height))
    axes = axes.flatten()
    
    for idx, slice_idx in enumerate(slice_indices):
        ax = axes[idx]
        # Extract the 2D slice
        slice_data = mask_data[:, :, slice_idx]
        
        # Create an RGB image for the slice
        rgb_image = np.zeros(slice_data.shape + (3,), dtype=float)
        
        # Assign colors to each ROI in the slice
        for label, color in label_to_color.items():
            mask = slice_data == label
            for c in range(3):  # RGB channels
                rgb_image[:, :, c] += mask * color[c]
        
        # Ensure values are within [0,1]
        rgb_image = np.clip(rgb_image, 0, 1)
        
        # Display the slice
        ax.imshow(rgb_image, origin='lower')
        ax.set_title(f"Slice {slice_idx}", fontsize=50)
        ax.axis('off')
    
    # Remove any unused subplots
    for j in range(idx + 1, grid_size[0]*grid_size[1]):
        fig.delaxes(axes[j])
    
    # ----------------------------
    # Step 6: Create Legend
    # ----------------------------
    # Create legend handles
    legend_handles = []
    for label, color in label_to_color.items():
        roi_name = label_list[label] if label < len(label_list) else f"Label {label}"
        handle = mpatches.Patch(color=color, label=roi_name)
        legend_handles.append(handle)
    
    # Place the legend outside the grid
    fig.legend(handles=legend_handles, loc='upper right', bbox_to_anchor=(1.15, 1), title="Brain ROIs", fontsize=35)
    
    # Adjust layout to accommodate legend and increase spacing between subplots
    plt.tight_layout(rect=[0, 0, 0.85, 1], w_pad=2, h_pad=2)
    
    # ----------------------------
    # Step 7: Save or Show the Figure
    # ----------------------------
    if output_file:
        try:
            # Ensure the output directory exists
            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            plt.savefig(output_file, dpi=dpi, bbox_inches='tight')
            print(f"Figure saved to '{output_file}' with DPI={dpi}.")
        except Exception as e:
            print(f"Error saving figure to '{output_file}': {e}")
    else:
        plt.show()
    
    plt.close(fig)  # Close the figure to free memory


# Specify the subject ID you want to visualize
subject_id = 'subj01'  # Change to the desired subject

# Define paths to the combined mask and label list
combined_mask_dir = "/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/"
combined_mask_file = os.path.join(combined_mask_dir, f"{subject_id}_combined_mask_nsd_general.nii.gz")
label_list_file = os.path.join(combined_mask_dir, f"{subject_id}_labels_nsd_general.txt")

# Check if files exist
if not os.path.exists(combined_mask_file):
    print(f"Combined mask file '{combined_mask_file}' does not exist.")
elif not os.path.exists(label_list_file):
    print(f"Label list file '{label_list_file}' does not exist.")
else:
    # Visualize the combined mask
    visualize_combined_mask(
        combined_mask_path=combined_mask_file,
        label_list_path=label_list_file,
        num_slices=25,
        grid_size=(5,5),
        output_file=os.path.join(combined_mask_dir, f"{subject_id}_combined_mask_visualization_nsd_general.png"),
        dpi=200  # Increased DPI for higher resolution
    )


Loaded combined mask from '/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/subj01_combined_mask_nsd_general.nii.gz' with shape (81, 104, 83).
Loaded 44 labels from '/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/subj01_labels_nsd_general.txt'.
Assigned 43 ROIs with repeated colors from the colormap.
Extracting 25 slices at indices: [10 12 15 17 20 23 25 28 31 33 36 38 41 44 46 49 52 54 57 59 62 65 67 70
 73]
Figure saved to '/home/naxos2-raid25/kneel027/home/kneel027/MindEye_Imagery/dataset/combined_masks/subj01_combined_mask_visualization_nsd_general.png' with DPI=200.
